# 라이브러리 설치



In [1]:
# !pip install pyLDAvis
# !pip install tensorflow

In [2]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
sklearn.set_config(display='text')

# 분석 과정

120다산콜재단은 2007년 다산콜센터로 시작한 서울시의 행정 상담 민원 서비스로 365일 24시간 상담 서비스를 제공하고 있다. 다산콜재단의 질문과 답변 데이터를 사용한다.

## 분석 과정은 크게 다음 두 가지 순서로 진행한다.

① 120다산콜재단 데이터(seoul-120-text.csv)를 머신러닝 방법인 잠재 디리클레 할당을 통한 토픽별로 분석 시각화  
&nbsp;&nbsp;&nbsp;▶ pyLDAvis 라이브러리를 이용한 시각화  
&nbsp;&nbsp;&nbsp;▶ TF-IDF 잠재 디리클레 할당  
② 토픽 모델링으로 데이터를 토픽별로 분류한 뒤, 딥러닝 방법 중 하나인 순환 신경망(RNN) 모델인 LSTM 모델로 학습해서 분류한다.

먼저 `잠재 디리클레 할당(Latent Dirichlet Allocation, LDA)`을 통한 토픽 모델링(Topic Modeling)으로 분석하고, 토픽 모델링을 시각화해 주는 pyLDAvis 라이브러리를 이용해 시각화해 본다. 그리고 학습, 테스트 데이터를 분리해서 LSTM 모델을 만들어 학습시킨다.

# 잠재 디리클레 할당으로 토픽 분류하기

LDA는 주어진 문서에 대해 각 문서에 어떤 토픽(주제)들이 있는지 서술하는 확률적 토픽 분류 기법 중 하나이다. 미리 알고 있는 주제별 단어 수 분포를 바탕으로, 주어진 문서에서 발견된 단 수 분포를 분석함으로써 해당 문서가 어떤 주제들을 함께 다루고 있을지 예측한다.

<img src="./잠재디리클레할당그림.png" width="900" align="left" />

LDA는 문서에 대한 범주의 연관성을 찾는 데 사용하는 확률론적 모델이며, 다음 두 가지 확률값을 사용해 문서를 군집화 한다.

P(단어|주제): 특정 단어가 주제와 연관될 확률, 이 첫 번째 확률 집합은 워드 * 주제 행렬로 간주된다.  
P(주제|문서): 문서와 관련된 항목, 이 두 번째 확률 집합은 주제 * 문서 행렬로 간주된다.

확률값은 모든 단어, 주제 및 문서에 대해 계산된다.

## 데이터 불러오기

In [3]:
df = pd.read_csv('./data/seoul-120-text.csv')
df.shape

(2645, 5)

In [4]:
df.head()

,번호,분류,제목,내용,내용번호
0,2645,복지,아빠 육아휴직 장려금,아빠 육아휴직 장려금 업무개요 남성근로자의 육아휴직을 장려하고 양육에 따른 경...,23522464
1,2644,경제,[서울산업진흥원] 서울메이드란?,서울산업진흥원 서울메이드란 서울의 감성을 담은 다양하고 새로운 경험을 제공하기 위해...,23194045
2,2643,환경,(강북구) 정비중,강북구 정비중 업무개요 투명 폐트병을 교환보상하므로 수거율을 높이고 폐기물을 감...,23032485
3,2642,복지,"광진맘택시 운영(임산부,영아 양육가정 전용 택시)",광진맘택시 운영임산부영아 양육가정 전용 택시 업무개요 교통약자인 임산부와 영아가정...,22904492
4,2641,복지,마포 뇌병변장애인 비전센터,마포 뇌병변장애인 비전센터 마포뇌병변장애인 비전센터 운영 구분 내용 목적 학...,22477798


In [5]:
df.isnull().sum()

번호      0
분류      0
제목      0
내용      0
내용번호    0
dtype: int64

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2645 entries, 0 to 2644
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   번호      2645 non-null   int64
 1   분류      2645 non-null   str  
 2   제목      2645 non-null   str  
 3   내용      2645 non-null   str  
 4   내용번호    2645 non-null   int64
dtypes: int64(2), str(3)
memory usage: 103.4 KB


In [7]:
df.dropna(inplace=True)

제목과 내용을 함께 사용해서 문서를 만들기 위해서 제목과 내용을 사이에 공백을 한 칸 넣어서 '+'로 연결해서 '문서'라는 파생 변수를 추가한다.

In [8]:
df['문서'] = df['제목'] + ' ' + df['내용']
df.head()

,번호,분류,제목,내용,내용번호,문서
0,2645,복지,아빠 육아휴직 장려금,아빠 육아휴직 장려금 업무개요 남성근로자의 육아휴직을 장려하고 양육에 따른 경...,23522464,아빠 육아휴직 장려금 아빠 육아휴직 장려금 업무개요 남성근로자의 육아휴직을 장...
1,2644,경제,[서울산업진흥원] 서울메이드란?,서울산업진흥원 서울메이드란 서울의 감성을 담은 다양하고 새로운 경험을 제공하기 위해...,23194045,[서울산업진흥원] 서울메이드란? 서울산업진흥원 서울메이드란 서울의 감성을 담은 다양...
2,2643,환경,(강북구) 정비중,강북구 정비중 업무개요 투명 폐트병을 교환보상하므로 수거율을 높이고 폐기물을 감...,23032485,(강북구) 정비중 강북구 정비중 업무개요 투명 폐트병을 교환보상하므로 수거율을 ...
3,2642,복지,"광진맘택시 운영(임산부,영아 양육가정 전용 택시)",광진맘택시 운영임산부영아 양육가정 전용 택시 업무개요 교통약자인 임산부와 영아가정...,22904492,"광진맘택시 운영(임산부,영아 양육가정 전용 택시) 광진맘택시 운영임산부영아 양육가정..."
4,2641,복지,마포 뇌병변장애인 비전센터,마포 뇌병변장애인 비전센터 마포뇌병변장애인 비전센터 운영 구분 내용 목적 학...,22477798,마포 뇌병변장애인 비전센터 마포 뇌병변장애인 비전센터 마포뇌병변장애인 비전센터 운영...


## BOW를 사용한 단어 벡터화

단어 토큰을 생성하고 각 단어의 개수를 세어 BOW 인코딩 벡터를 생성한다.

단어들의 출현 빈도(frequency)로 여러 문서를 벡터화하기 위해 CountVectorizer를 import 한다.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

In [10]:
cv = CountVectorizer(stop_words=['돋움', '경우', '또는'])
cv

CountVectorizer(stop_words=['돋움', '경우', '또는'])

fit_transform() 메소드로 문장에서 노출되는 feature(문서의 특징이 될 만한 단어) 문서 단어 희소 행렬(dtm_cv) 생성한다.

In [11]:
dtm_cv = cv.fit_transform(df['문서'])
dtm_cv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 118578 stored elements and shape (2645, 56651)>

In [12]:
dtm_cv.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(2645, 56651))

In [13]:
vocab = cv.get_feature_names_out()
vocab

array(['03월', '08년', '10', ..., '힘쓴다', '힘을', '힘이'],
      shape=(56651,), dtype=object)

CountVectorizer 모델의 vocabulary_ 속성으로 어떤 단어들의 집합이 있는지 딕셔너리 형태로 확인할 수 있다.

In [14]:
cv.vocabulary_

{'아빠': 30166,
 '육아휴직': 35794,
 '장려금': 40098,
 '업무개요': 31494,
 '남성근로자의': 9780,
 '육아휴직을': 35798,
 '장려하고': 40101,
 '양육에': 31079,
 '따른': 14184,
 '경제적': 3650,
 '부담을': 20458,
 '완화함으로써': 33605,
 '일과': 37802,
 '가정생활의': 694,
 '양립': 31009,
 '가족친화적인': 784,
 '사회환경': 22911,
 '조성': 43603,
 '지원대상': 46358,
 '신청일': 29324,
 '기준': 8969,
 '이상': 36691,
 '계속하여': 3781,
 '서초구에': 24773,
 '주민등록': 44417,
 '되어': 13584,
 '있는': 38959,
 '육아휴직자': 35799,
 '대상자녀': 11880,
 '신청기간': 29232,
 '시작일': 28426,
 '이후': 37282,
 '개월부터': 1767,
 '종료일': 44002,
 '개월': 1753,
 '이내': 36439,
 '신청방법': 29260,
 '온라인': 33390,
 '서초구청': 24776,
 '홈페이지': 55328,
 '경로': 3421,
 '분야별정보': 21117,
 '복지': 20129,
 '영유아복지': 32834,
 '아빠육아휴직장려금': 30170,
 '신청': 29215,
 '바로가기': 17408,
 '방문': 18118,
 '동주민센터': 13431,
 '여성보육과': 31951,
 '구비서류': 6902,
 '고용센터': 4208,
 '발행': 18048,
 '육아휴직급여': 35795,
 '지급결정': 45803,
 '통지서': 51047,
 '주민등록등본': 44419,
 '부세대원': 20662,
 '이름과': 36609,
 '전입일자': 41564,
 '포함': 52225,
 '모든': 15657,
 '구성원': 6951,
 '주민번호': 44460,
 '뒷자리': 13756,
 '

벡터를 표현하려면 단어 가방에 있는 모든 단어를 행렬값으로 나타내야 한다. toarray() 메소드로 희소 행렬을 넘파이 배열로 변환하고 get_feature_names_out() 메소드를 실행해서 불러온 단어 목록을 데이터프레임으로 만들어서 확인한다.

In [15]:
pd.DataFrame(dtm_cv.toarray(), columns=vocab)

,03월,08년,10,100명이상인,100세가,10만원,10만원상당,10명이고,10인승,10인의,...,힐링프로그램을,힐링하는,힐스테이트,힘들,힘들경우,힘들고,힘쓰고있습니다,힘쓴다,힘을,힘이
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2640,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2641,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2642,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2643,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


희소 행렬을 단어 목록별로 출현 횟수를 sum() 메소드로 계산해서 출현 횟수의 오름차순으로 정렬한다.  
오름차순으로 정렬한 결과를 보면 아랫부분에 '돋움', '있는', '있습니다' 같은 고빈도 단어들이 있다. 이러한 단어들은 불용어로 다룰 것인지 고려해야 한다.

In [16]:
pd.DataFrame(dtm_cv.toarray(), columns=vocab).sum().sort_values()

힐스테이트        1
힐링하는         1
힐링프로그램을      1
18세          1
19           1
          ... 
대한         394
서울시        578
어떻게        597
있습니다       685
있는         718
Length: 56651, dtype: int64

## 잠재 디리클레 할당 적용하기

정답인 '분류'의 유일한 값을 확인해서 토픽의 개수를 확인하자.

In [17]:
print(df.분류.value_counts())
NUM_TOPICS = len(set(df['분류']))
print(NUM_TOPICS)

분류
행정        1098
경제         823
복지         217
환경         124
주택도시계획     110
문화관광        96
교통          90
안전          51
건강          23
여성가족        13
Name: count, dtype: int64
10


각 문서에 어떤 주제들이 있는지 확인하는 사이킷런의 잠재 디리클레 분석을 사용하기 위해서 LatentDirichletAllocation를 import 한다.

In [18]:
from sklearn.decomposition import LatentDirichletAllocation

n_components 하이퍼파라미터에 분석할 토픽의 개수(기본값 10)를 설정하고 모델을 만든다. 재현성(매번 똑같은 결과)을 위해 random_state를 설정했다.

In [19]:
LDA_model = LatentDirichletAllocation(n_components=NUM_TOPICS, random_state=42)
LDA_model

LatentDirichletAllocation(random_state=42)

fit() 메소드로 작성된 LDA 모델에 문서 단어 행렬(dtm_cv)을 넣어 학습시킨다.

In [20]:
LDA_model.fit(dtm_cv)

LatentDirichletAllocation(random_state=42)

# pyLDAvis를 통한 LDA 토픽 모델 시각화

pyLDAvis는 파이썬의 토픽 모델링을 구현해 주는 좋은 도구로 사용자 말뭉치의 토픽을 자동으로 추출해 해석하고 대화형 웹 기반으로 시각화할 수 있도록 설계되었다. 시각화 결과는 IPython 노트북에서 사용하기 위한 것이지만, 독립 실행형 HTML 파일로 저장해서 쉽게 공유할 수 있다.

어떤 방법으로 전처리와 모데링을 해 주느냐에 따라 pyLDAvis로 토픽 모델링한 결과 또한 다르게 나타난다. 다음은 BOW 방식으로 벡터화했을 때의 결과이다. 모델링에서 최적화한 토픽별 대표 단어들을 반환한 뒤 t-SNE(t-Distributed Stochastic Neighbor Embedding)을 통해 고차원 데이터를 2차원으로 차원 축소해 시각화 한다. t-SNE는 고차원 데이터를 2D나 3D 공간으로 줄여 눈으로 직접 확인할 수 있게 만드는 대표적인 시각화용 차원 축소 알고리즘이다.

LDA 토픽 모델을 시각화하기 위해서 pyLDAvis를 설치하고 import 한다.  
pyLDAvis 버전 `3.4.0`부터 라이브러리 내부 구조가 `변경`돼서 기존에 사용하던 pyLDAvis.sklearn 대신에 `pyLDAvis.lda_model`를 사용해야 한다.

In [21]:
import pyLDAvis.lda_model

prepare() 메소드로 LDA 토픽 모델링을 시각화 한다.  
`lda_model`: 학습이 완료된 LDA 토픽 모델링 객체를 지정한다.  
`dtm`: CountVectorizer나 TfidfVectorizer를 통해 변환된 문서 단어 행렬 객체를 지정한다.  
`vectorizer`: 2번째 인수인 `dtm` 생성할 때 사용한 벡터라이저 객체를 지정한다.  
`mds`: 시각화에 사용할 다차원 축소 알고리즘(Multi-Dimensional Scaling)을 지정한다. `pcoa`가 기본값이고 Principal Component Analysis를 의미한다.

## LDA 토픽 모델링 시각화 결과를 IPython으로 출력하기

`pyLDAvis.enable_notebook()`  
`pyLDAvis.lda_model.prepare(lda_model=LDA_model, dtm=dtm_cv, vectorizer=cv, mds='tsne')`

## LDA 토픽 모델링 시각화 결과를 HTML 파일로 출력하기

save_html() 메소드를 사용하면 LDA 토픽 모델링 시각화 결과를 HTML 파일로 만든다.  
`data`: prepare() 메소드로 LDA 토픽 모델링 시각화 명령을 지정한다.  
`fileobj`: HTML 파일의 경로와 이름을 지정한다.

In [22]:
pyLDAvis.save_html(data=pyLDAvis.lda_model.prepare(lda_model=LDA_model, dtm=dtm_cv, vectorizer=cv, mds='tsne'), fileobj='./lda_result.html')

pyLDAvis로 시각화한 LDA 토픽 모델링 결과는 왼쪽의 지도와 오른쪽의 막대그래프로 구성된다.

<img src="./LDA토픽모델링시각화.png" width="1200" align="left" />

## 왼쪽 화면

토픽 간 거리 지도(Intertopic Distance Map), 전체 데이터셋에서 추출된 토픽들이 서로 어떤 관계를 맺고 있는지 보여준다.  
원의 크기: 해당 토픽이 전체 문서에서 차지하는 비중 원이 클수록 많은 문서에 다뤄진 주요 주제이다.  
원의 위치: 토픽 간의 유사성을 나타낸다.  
&nbsp;&nbsp;&nbsp;▶ 멀리 떨어져 있을수록 주제가 뚜렷하게 구분되는 독립적인 토픽이다. 이상적인 결과.  
&nbsp;&nbsp;&nbsp;▶ 많이 겹쳐 있을수록 두 토픽이 공유하는 단어가 많아 주제가 중복될 가능성이 크다. 토픽 개수를 줄여서 다시 모델링하는 것이 좋다.

## 오른쪽 화면

주요 단어 목록(Top-30 Most Salient Terms)으로 토픽 간 거리 지도에서 특정 원(토픽)을 클릭했을 때 나타는 단어들의 분포이다.  
▶ 하늘색 막대(Overall term frequency)는 해당 단어가 전체 데이터에서 나타는 총 빈도이다.  
▶ 빨간색 막대(Estimated term frequency within the selected topic)는 해당 단어가 선택한 토픽 내에서 나타나는 빈도이다.  
해석하는 방법은 하늘색 대비 빨간색 비중이 높을수록, 그 단어는 해당 토픽을 정의하는 핵심 키워드라고 볼 수 있다.

## 오른쪽 상단의 슬라이더

람다($\lambda$)값 조절  
$\lambda$값을 조절하면 단어 리스트의 정렬 기준이 바뀐다. 토픽의 이름을 정의할 때 매우 중요하다.  
$\lambda$를 1로 설정하면 토픽 내 빈도수가 높은 순서로 보여준다.  
$\lambda$를 0으로 설정하면 다른 토픽에는 거의 없고 해당 토픽에서만 독보적으로 등장하는 단어 위주로 보여준다. 토픽의 정체성 파악하기 좋다.  
$\lambda$의 추천 설정은 0.6으로 일반적으로 가장 해석이 쉬운 중첩 지점이다. 이 설정에서 상위 5 ~ 10개 단어를 보고 토픽 이름을 결정한다.

In [23]:
vis_data = pyLDAvis.lda_model.prepare(LDA_model, dtm_cv, cv, mds='tsne')
topic_df = vis_data.topic_info
topic_df.head()

,Term,Freq,Total,Category,logprob,loglift
119,amp,63.0,63.0,Default,30.0,30.0
39480,자본금은,52.0,52.0,Default,29.0,29.0
36388,의한,205.0,205.0,Default,28.0,28.0
8795,기술능력과,34.0,34.0,Default,27.0,27.0
31204,어떻게,441.0,441.0,Default,26.0,26.0


pyLDAvis에서 생성되는 topic_df(vis_data.topic_info) 데이터프레임의 Category 열은 시각화 화면의 우측 바 차트에 출력할 단어들의 필터링 상태를 나타낸다.  
Category 열에는 데이터프레임 전체 행에 걸쳐 크게 두 가지 형태의 문자열 값이 저장된다.  
'Default'는 특정 토픽을 선택하지 않은 전체 데이터셋 상태를 나타낸다.  
'Topic1', 'Topic2', 'Topic3', ... 는 각 개별 토픽 상태를 나타낸다.

In [24]:
for i in range(1, LDA_model.n_components + 1):
    topic_keywords = topic_df[topic_df.Category == f'Topic{i}'].head(10)['Term'].values
    print(f'Topic{i}: {", ".join(topic_keywords)}')

Topic1: 자본금은, 기술능력과, 건설기술자, 바닥면적의, 관련종목의, 제종근린생활시설, 기술자격취득자, 대기환경보전법에, 되나요건설기술관리법에, 용도에
Topic2: 장기기증, 현금영수증, 여론조사, 트레킹, 시가지, 여성이룸센터, 지그재그, 제강, 청원, 환자안심병원
Topic3: amp, apos, 체험요금, 임신출산, 정읍, 수련관, 맘편한카드, 차상위, 신상정보, 무안
Topic4: 급식, 용인, 교복, 예체능, 챌린지, 서북병원, 세대주가, 청약부금, 청약예금, 금주
Topic5: 원어민, 영어회화, 특별활동, 승합, 화물, 잠수교, 신체활동리더, 승용, 상시근로자수, 홍제천
Topic6: 층평일, 주민대상, 서울함, 권리가액이, 외국인주민, 한강의, 이색달리기, 분양받을, 증강현실, 분양건축물의
Topic7: 치매지원센터, 의료기관이, 구의아리수정수센터, 서울상, 서울특별시육아종합지원센터, 후계농업경영인, 봉수대, 자격정지, 보육서비스, 콘테스트
Topic8: 나와서, 버스이용, 계약심사, 천세대, 일은, 전달합니다, 상대방의, 중계사가, 통화내용을, 주택단지의
Topic9: 기획예산과, 전산정보과, 빈집, 개인회생, 제조구매, 주택용, 처리한, 친구의, 광화문시민열린마당, 남산동
Topic10: 시간외, 서커스, 이행기준, 심사요청, 재배정사업, 관리규약에, 구성원의, 점자스티커, 길음동, 스트리트


## TF-IDF를 사용한 단어 벡터화

단어들의 출현 빈도를 TF-IDF 방식으로 단어의 가중치를 조정한 여러 문서를 벡터화하기 위해 TfidfVectorizer를 import 한다.

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [26]:
tfidf = TfidfVectorizer(stop_words=['돋움', '경우', '또는', '있습니다', '있는', '합니다'])
tfidf

TfidfVectorizer(stop_words=['돋움', '경우', '또는', '있습니다', '있는', '합니다'])

fit_transform() 메소드로 문장에서 노출되는 feature(문서의 특징이 될 만한 단어) 문서 단어 희소 행렬(dtm_tfidf) 생성한다.

In [27]:
dtm_tfidf = tfidf.fit_transform(df['문서'])
dtm_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 117379 stored elements and shape (2645, 56648)>

In [28]:
dtm_tfidf.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(2645, 56648))

In [29]:
vocab = tfidf.get_feature_names_out()
vocab

array(['03월', '08년', '10', ..., '힘쓴다', '힘을', '힘이'],
      shape=(56648,), dtype=object)

In [30]:
tfidf.vocabulary_

{'아빠': 30166,
 '육아휴직': 35794,
 '장려금': 40096,
 '업무개요': 31494,
 '남성근로자의': 9780,
 '육아휴직을': 35798,
 '장려하고': 40099,
 '양육에': 31079,
 '따른': 14184,
 '경제적': 3650,
 '부담을': 20458,
 '완화함으로써': 33605,
 '일과': 37802,
 '가정생활의': 694,
 '양립': 31009,
 '가족친화적인': 784,
 '사회환경': 22911,
 '조성': 43601,
 '지원대상': 46356,
 '신청일': 29324,
 '기준': 8969,
 '이상': 36691,
 '계속하여': 3781,
 '서초구에': 24773,
 '주민등록': 44415,
 '되어': 13584,
 '육아휴직자': 35799,
 '대상자녀': 11880,
 '신청기간': 29232,
 '시작일': 28426,
 '이후': 37282,
 '개월부터': 1767,
 '종료일': 44000,
 '개월': 1753,
 '이내': 36439,
 '신청방법': 29260,
 '온라인': 33390,
 '서초구청': 24776,
 '홈페이지': 55325,
 '경로': 3421,
 '분야별정보': 21117,
 '복지': 20129,
 '영유아복지': 32834,
 '아빠육아휴직장려금': 30170,
 '신청': 29215,
 '바로가기': 17408,
 '방문': 18118,
 '동주민센터': 13431,
 '여성보육과': 31951,
 '구비서류': 6902,
 '고용센터': 4208,
 '발행': 18048,
 '육아휴직급여': 35795,
 '지급결정': 45801,
 '통지서': 51045,
 '주민등록등본': 44417,
 '부세대원': 20662,
 '이름과': 36609,
 '전입일자': 41562,
 '포함': 52223,
 '모든': 15657,
 '구성원': 6951,
 '주민번호': 44458,
 '뒷자리': 13756,
 '미포함': 17033,
 

In [31]:
pd.DataFrame(dtm_tfidf.toarray(), columns=vocab)

,03월,08년,10,100명이상인,100세가,10만원,10만원상당,10명이고,10인승,10인의,...,힐링프로그램을,힐링하는,힐스테이트,힘들,힘들경우,힘들고,힘쓰고있습니다,힘쓴다,힘을,힘이
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2640,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2641,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2642,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2643,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
dist = np.sum(dtm_tfidf, axis=0)
pd.DataFrame(dist, columns=vocab).T.sort_values(by=0).tail(10)

,0
의한,15.021840
무엇입니까,15.270257
이상,15.577954
관한,16.593598
무엇인가요,16.650743
따라,16.652594
대한,18.866037
있나요,19.707343
서울시,22.586695
어떻게,37.924574


## 유사도 분석하기

문의 내용을 보면 비슷한 주제는 비슷한 위치체 놓인다. 따라서 벡터화된 텍스트의 거리를 측정하면 어떤 텍스트가 가까운 위치에 있는지를 계산할 수 있다. 문의 내용을 확인하고 등장 빈도에 기반해, 코사인 유사도 알고리즘 적용한다.

사이킷런은 코사인 유사도를 계산하는 기능을 제공한다. 코사인 유사도는 두 벡터 간의 각도의 코사인 값을 사용하여 두 벡터가 얼마나 같은 방향을 가리키고 있는지를 측정하는 유사도 지표이다.

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
similarity_simple_pair = cosine_similarity(dtm_tfidf[0], dtm_tfidf)
result_list = similarity_simple_pair.tolist()[0]
result_list

[1.0000000000000002,
 0.0,
 0.009923678841957665,
 0.04840410811630889,
 0.020255740517537653,
 0.01854578524134521,
 0.007689200514538419,
 0.01852208319048694,
 0.004388306893623124,
 0.015596366271907504,
 0.006417818542155354,
 0.004801755890490625,
 0.022530745285606658,
 0.010670974912200707,
 0.005465729535177704,
 0.007180867167060437,
 0.0,
 0.007191459219804233,
 0.0,
 0.005548948853251886,
 0.0,
 0.014999956175567453,
 0.01346177710130625,
 0.03186585941980262,
 0.006324895844168201,
 0.011305577503097426,
 0.01042721964765217,
 0.0027107806748704246,
 0.02811621396063277,
 0.01945708490794608,
 0.009451308855535759,
 0.0,
 0.0,
 0.003921239868317877,
 0.03577777313658556,
 0.04603660879021193,
 0.0021097618250367593,
 0.002429415475842772,
 0.013110061818089973,
 0.014356080256311323,
 0.006058250439663021,
 0.0025792085469002564,
 0.006370413664274528,
 0.0026519404815570063,
 0.0,
 0.005011121517701414,
 0.002377621300635238,
 0.0,
 0.017201197098464488,
 0.00596595158453

result_list를 '유사도' 파생 변수로 생성하고 유사도가 높은 순으로 정렬한다.  
유사도가 높은 문서는 거리가 가깝다고 불 수 있으므로, 이 방법을 응용하면 문의에 대한 비슷한 질문을 추천해 보여주는 등과 같이 활용할 수 있다.

In [47]:
df['유사도'] = result_list
df.sort_values(by='유사도', ascending=False).head(10)

,번호,분류,제목,내용,내용번호,문서,유사도
0,2645,복지,아빠 육아휴직 장려금,아빠 육아휴직 장려금 업무개요 남성근로자의 육아휴직을 장려하고 양육에 따른 경...,23522464,아빠 육아휴직 장려금 아빠 육아휴직 장려금 업무개요 남성근로자의 육아휴직을 장...,1.000000
1772,873,경제,도시계획시설부지 재결신청 이후 진행단계는 어떤 과정을 거칩니까?,도시계획시설부지 재결신청 이후 진행단계는 어떤 과정을 거칩니까재결신청 이후 재결신청...,2897109,도시계획시설부지 재결신청 이후 진행단계는 어떤 과정을 거칩니까? 도시계획시설부지 재...,0.057158
850,1795,경제,주민대표회의 구성원 몇명입니까?,주민대표회의 구성원 몇명입니까인이상 인 이하로 구성합니다,2896070,주민대표회의 구성원 몇명입니까? 주민대표회의 구성원 몇명입니까인이상 인 이하로 구성합니다,0.056152
539,2106,행정,행려자도 아니고 시설수용자도 아닌 사람이 살고 있던 비닐하우스에서 화상을 입었습니다...,행려자도 아니고 시설수용자도 아닌 사람이 살고 있던 비닐하우스에서 화상을 입었습니다...,2896135,행려자도 아니고 시설수용자도 아닌 사람이 살고 있던 비닐하우스에서 화상을 입었습니다...,0.051952
3,2642,복지,"광진맘택시 운영(임산부,영아 양육가정 전용 택시)",광진맘택시 운영임산부영아 양육가정 전용 택시 업무개요 교통약자인 임산부와 영아가정...,22904492,"광진맘택시 운영(임산부,영아 양육가정 전용 택시) 광진맘택시 운영임산부영아 양육가정...",0.048404
155,2490,경제,[농업기술센터] 후계농업경영인 선정 및 청년창업형 후계농업경영인 신청 안내,농업기술센터 후계농업경영인 선정 및 청년창업형 후계농업경영인 신청 안내 업무...,2896492,[농업기술센터] 후계농업경영인 선정 및 청년창업형 후계농업경영인 신청 안내 농업기술...,0.046280
35,2610,행정,[시ㆍ구정외 타기관 관련 상담] 고용노동부 [일자리 안정자금],시구정외 타기관 관련 상담 고용노동부 일자리 안정자금 최저임금 해결사 일자리 안정자...,14425328,[시ㆍ구정외 타기관 관련 상담] 고용노동부 [일자리 안정자금] 시구정외 타기관 관련...,0.046037
141,2504,경제,[농업기술센터] 도시농업전문가양성교육 신청,농업기술센터 도시농업전문가양성교육 신청 업무개요 도시농업육성 및 지원에 관한 법...,2897619,[농업기술센터] 도시농업전문가양성교육 신청 농업기술센터 도시농업전문가양성교육 신청 ...,0.043873
174,2471,행정,찾아가는 아버지교실,찾아가는 아버지교실 업무개요 아버지와 자녀 간 공감대 형성을 통해 가족...,2898611,찾아가는 아버지교실 찾아가는 아버지교실 업무개요 아버지와 자녀 간 공감...,0.043034
233,2412,경제,[농업기술센터] 귀농창업 평일반 교육 신청,농업기술센터 귀농창업 평일반 교육 신청 업무개요 서울시민을 대상으로 귀농창업...,2898211,[농업기술센터] 귀농창업 평일반 교육 신청 농업기술센터 귀농창업 평일반 교육 신청 ...,0.041945


# 순환 신경망으로 텍스트 분류하기

순환 신경망(Recurrent Neural Network, RNN)은 이름에서 알 수 있듯이 신경망을 사용해 모델링한다. 순환 신경망의 가장 큰 특징은 이전 단어가 다음 단어에 의존 관계를 지닐 수 있도록 시퀀스 형태로 입력된다는 것이다.

## 데이터 불러오기

In [48]:
df = pd.read_csv('./data/seoul-120-text.csv')
df.shape

(2645, 5)

In [49]:
df['문서'] = df['제목'] + ' ' + df['내용']
df.head()

,번호,분류,제목,내용,내용번호,문서
0,2645,복지,아빠 육아휴직 장려금,아빠 육아휴직 장려금 업무개요 남성근로자의 육아휴직을 장려하고 양육에 따른 경...,23522464,아빠 육아휴직 장려금 아빠 육아휴직 장려금 업무개요 남성근로자의 육아휴직을 장...
1,2644,경제,[서울산업진흥원] 서울메이드란?,서울산업진흥원 서울메이드란 서울의 감성을 담은 다양하고 새로운 경험을 제공하기 위해...,23194045,[서울산업진흥원] 서울메이드란? 서울산업진흥원 서울메이드란 서울의 감성을 담은 다양...
2,2643,환경,(강북구) 정비중,강북구 정비중 업무개요 투명 폐트병을 교환보상하므로 수거율을 높이고 폐기물을 감...,23032485,(강북구) 정비중 강북구 정비중 업무개요 투명 폐트병을 교환보상하므로 수거율을 ...
3,2642,복지,"광진맘택시 운영(임산부,영아 양육가정 전용 택시)",광진맘택시 운영임산부영아 양육가정 전용 택시 업무개요 교통약자인 임산부와 영아가정...,22904492,"광진맘택시 운영(임산부,영아 양육가정 전용 택시) 광진맘택시 운영임산부영아 양육가정..."
4,2641,복지,마포 뇌병변장애인 비전센터,마포 뇌병변장애인 비전센터 마포뇌병변장애인 비전센터 운영 구분 내용 목적 학...,22477798,마포 뇌병변장애인 비전센터 마포 뇌병변장애인 비전센터 마포뇌병변장애인 비전센터 운영...


In [50]:
df.분류.value_counts()

분류
행정        1098
경제         823
복지         217
환경         124
주택도시계획     110
문화관광        96
교통          90
안전          51
건강          23
여성가족        13
Name: count, dtype: int64

행정의 빈도수와 건강, 여성가족의 빈도수 차이가 심하다. 분류별 빈도수 값이 불균형이 심할 경우 전체 데이터로 예측하면 성능이 떨어질 수 있으므로, 언더샘플링이나 오버샘플링으로 정답값을 균형있게 만들어 주기도 한다. 여기에서는 상위 데이터 세 개만 사용해 분석한다. 이렇게 일부 분류만 사용하더라도 정답에 불균형이 있기 때문에 빈도가 많은 분류의 예측 정확도가 더 높게 나올 것이다.

df 데이터프레임의 '분류' 열에는 10종류의 분류가 있는데 '행정', '경제', '복지'만 추출하려면 아래와 같이 OR(|) 연산자를 사용하면 된다.  
`df[(df.분류 == '행정') | (df.분류 == '경제') | (df.분류 == '복지')]`  
하지만 추출하려는 열이 매우 많다면 OR(|) 연산자를 사용하면 코드가 많이 길어진다. 이럴 경우 `isin()` 메소드를 사용하면 논리식에 `in` 연산자를 사용하는 것과 같은 효과를 낼 수 있다.

In [64]:
df = df[df.분류.isin(['행정', '경제', '복지'])]
df.shape

(2138, 6)

In [65]:
df.head()

,번호,분류,제목,내용,내용번호,문서
0,2645,복지,아빠 육아휴직 장려금,아빠 육아휴직 장려금 업무개요 남성근로자의 육아휴직을 장려하고 양육에 따른 경...,23522464,아빠 육아휴직 장려금 아빠 육아휴직 장려금 업무개요 남성근로자의 육아휴직을 장...
1,2644,경제,[서울산업진흥원] 서울메이드란?,서울산업진흥원 서울메이드란 서울의 감성을 담은 다양하고 새로운 경험을 제공하기 위해...,23194045,[서울산업진흥원] 서울메이드란? 서울산업진흥원 서울메이드란 서울의 감성을 담은 다양...
3,2642,복지,"광진맘택시 운영(임산부,영아 양육가정 전용 택시)",광진맘택시 운영임산부영아 양육가정 전용 택시 업무개요 교통약자인 임산부와 영아가정...,22904492,"광진맘택시 운영(임산부,영아 양육가정 전용 택시) 광진맘택시 운영임산부영아 양육가정..."
4,2641,복지,마포 뇌병변장애인 비전센터,마포 뇌병변장애인 비전센터 마포뇌병변장애인 비전센터 운영 구분 내용 목적 학...,22477798,마포 뇌병변장애인 비전센터 마포 뇌병변장애인 비전센터 마포뇌병변장애인 비전센터 운영...
5,2640,행정,2021년도 중1·고1 신입생 입학준비금 지원,년도 중고 신입생 입학준비금 지원 업무개요 서울시는 전국 최초로 년도부터 개 자...,22227896,2021년도 중1·고1 신입생 입학준비금 지원 년도 중고 신입생 입학준비금 지원 ...


## 학습 데이터셋과 테스트 데이터셋 분리하기

데이터프레임에서 독립 변수(문제, x_data)와 종속 변수(정답, y_data)를 추출한다.

In [77]:
x_data, y_data = df.문서, df.분류
# print(x_data, y_data)

## 레이블 값을 원-핫 인코딩하기

get_dummies() 메소드를 사용해서 레이블값을 0과 1로 이루어진 행렬 형태로 만든다. 이를 원-핫 인코딩(One-Hot Encoding)이라고 하는데 데이터를 0과 1로 구별한다. 고유값에 해당되는 단어는 1(True), 나머지는 0(False)으로 만드는 방식으로, 사이킷런 머신러닝 알고리즘에는 문자열값을 넣을 수 없기 때문에 숫자형으로 인코딩한 뒤 학습시키기 위해서이다.

In [83]:
y_onehot = pd.get_dummies(y_data, dtype=int)
y_onehot.head()

,경제,복지,행정
0,0,1,0
1,1,0,0
3,0,1,0
4,0,1,0
5,0,0,1


데이터프레임에서 추출된 독립 변수와 종속 변수를 학습 데이터셋과 테스트 데이터셋으로 나누기 위해서 train_test_split를 import 한다.

In [84]:
from sklearn.model_selection import train_test_split

train_test_split() 함수는 독립 변수(x_data), 원-핫 인코딩된 종속 변수(y_onehot) 학습 데이터셋 80%, 테스트 데이터셋 20%의 비율로 분할한다. 이때, 재현성을 유지하기 위해 random_state를 지정했고 층화 추출을 수행하기 위해 stratify를 지정했다.  
층화 추출이란 종속 변수에 저장된 각 클래스의 비율을 계산해서 분할된 학습 데이터셋과 테스트 데이터셋에서 그 클래스 비율이 원본 비율과 동일하게 유지한다.

In [86]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_onehot, train_size=0.8, random_state=42, stratify=y_onehot)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(1710,) (428,) (1710, 3) (428, 3)


y_train, y_test의 값을 확인해 보면 원-핫 인코딩된 결과 즉, 0과 1로 이루어진 희소 행렬 형태로 이루어진 것을 확인할 수 있다.  
데이터셋을 나누다 보면 특정 분류는 학습 데이터셋에 많고, 테스트 데이터셋에는 너무 적어 균형있게 학습하지 못하는 현상이 발생되기도 한다. 이를 방지하기 위해 train_test_split() 함수로 데이터셋을 나눌 때 stratify 속성에 독립 변수(정답) 지정하면 학습 데이터셋과 테스트 데이터셋의 정답 비율을 맞춰서 나눠준다. 다음 코드를 실행해 보면 '경제', '복지', '행정'의 정답 비율이 비슷한 비율로 나뉘 것을 확인할 수 있다.

In [92]:
print(y_train.mean())
print(y_test.mean())

경제    0.384795
복지    0.101754
행정    0.513450
dtype: float64
경제    0.385514
복지    0.100467
행정    0.514019
dtype: float64


## 벡터화하기

토큰화하기  
토크나이저는 텍스트를 여러개의 토큰으로 나눈다. 케라스의 Tokenizer 클래스를 사용하면 각 텍스트를 일련의 정수(정수는 단어 사전의 토큰 인덱스) 또는 단어수에 따라 각 토큰의 계사가 이진일 수 있는 벡터로 변환해 테스트 데이터를 벡터화(Vectorization)할 수 있다.

시퀀스 만들기  
Tokenizer는 데이터에 출현하는 모든 단어의 계수를 세고 빈도수를 정렬해서 num_words 속성에 지정된 만큼만 숫자로 반환한다.

In [93]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [118]:
# 모델의 복잡도를 제어하고 메모리를 효율적으로 사용하기 위해서 데이터셋에 고유한 단어가 아무리 많아도 빈도수가 높은 상위 10,000개의 단어만 사용한다는 설정이다.
vocab_size = 10000
# Out-Of-Vocabulary(단어 사전에 없는 단어)를 처리할 특별 토큰을 지정한다.
# 단어 사전에 포함되지 않은(상위 10,000개에 못 든) 생소한 단어를 만나면 '<oov>'라는 고정된 값으로 치환한다. 이 처리를 하지 않으면 단어 사전에 포함되지 않은
# 단어는 그냥 삭제되어 문장의 길이가 변하거나 문맥이 왜곡될 수 있다.
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<oov>')
tokenizer

fit_on_texts() 메소드에 학습 데이터를 넘겨서 tensorflow의 Tokenizer를 학습시킨다.  
학습 결과는 단어와 인덱스가 key와 value로 구성된 딕셔너리를 생성하고 영문자인 경우 자동으로 소문자로 변환해서 반환되며, 느낌표나 마침표 같은 구두점은 자동으로 제거한다.

In [119]:
tokenizer.fit_on_texts(x_train)

word_index 속성으로 단어와 인덱스가 key, value로 이루어진 딕셔너리를 얻어온다.

In [125]:
print(tokenizer.word_index)

{'<oov>': 1, '및': 2, '돋움': 3, '수': 4, '경우': 5, '또는': 6, '등': 7, '있는': 8, '년': 9, '월': 10, '있습니다': 11, '어떻게': 12, '서울시': 13, '후': 14, '중': 15, '일': 16, '홈페이지': 17, '대한': 18, '그': 19, '따라': 20, '의한': 21, '위한': 22, '가능': 23, '이상': 24, '하는': 25, '관한': 26, '할': 27, '경우에는': 28, '있나요': 29, '등을': 30, '합니다': 31, '각': 32, '층': 33, '문의': 34, '시': 35, '위하여': 36, '신청': 37, '관련': 38, '있음': 39, '지원': 40, '등의': 41, '규정에': 42, '확인': 43, '개': 44, '운영': 45, '제조': 46, '필요한': 47, '됩니다': 48, '따른': 49, '내용': 50, '무엇인가요': 51, '무엇입니까': 52, '것': 53, '해당': 54, '어떤': 55, '하나요': 56, '통해': 57, '원': 58, '하고': 59, '등에': 60, '되나요': 61, '함': 62, '주소': 63, '교육': 64, '명': 65, '기타': 66, '서울특별시': 67, '이상의': 68, '위해': 69, '구분': 70, '안내': 71, '있으며': 72, '내': 73, '이용': 74, '대하여': 75, '공무원': 76, '회': 77, '어린이집': 78, '만원': 79, '시간': 80, '프로그램': 81, '평일': 82, '서울': 83, '말한다': 84, '당해': 85, '자': 86, '다른': 87, '없는': 88, '접수': 89, '직접': 90, '인': 91, '한': 92, '단': 93, '한다': 94, '기타사항': 95, '사업': 96, '건축물': 97, '있도록': 98, '업무개요': 99,

word_counts 속성으로 단어와 빈도수가 묶인 튜플을 얻어온다.

In [129]:
list(tokenizer.word_counts.items())[:10]

[('우리아이의', 2),
 ('배정', 11),
 ('초등학교를', 2),
 ('알고', 45),
 ('싶어요', 10),
 ('싶어요매년', 1),
 ('취학통지서', 1),
 ('발급', 66),
 ('때가', 1),
 ('되면', 7)]

단어별 빈도를 고빈도순으로 정렬한다.

In [138]:
word_df = pd.DataFrame(tokenizer.word_counts.items(), columns=['단어', '빈도수']).set_index('단어')
word_df.sort_values(by='빈도수', ascending=False).T

단어,및,돋움,수,경우,또는,등,있는,년,있습니다,월,...,굴착면,제거하였느가,안전히,부석은,발파후,낙석방지,옵니다,통보가,통학구역이,때가
빈도수,1455,1110,771,595,550,547,411,400,379,379,...,1,1,1,1,1,1,1,1,1,1


texts_to_sequences() 메소드로 fit_on_texts() 메소드가 벡터화한 정보에서 찾아서 문장을 숫자로 표현한 리스트로 만든다.

In [142]:
train_sequence = tokenizer.texts_to_sequences(x_train)
test_sequence = tokenizer.texts_to_sequences(x_test)

## 패딩하기

자연어 처리를 하다 보면 각 문장 또는 문서의 길이가 서로 다를 수 있다. 컴퓨터는 문서의 길이가 같아야만 하나의 행렬로 보고 한꺼번에 묶어서 처리할 수 있으므로, 병렬 연산을 위해서는 여러 문장의 길이를 동일하게 맞추는 작업이 필요하다.

<img src="./패딩적용.png" width="600" align="left" />

실제 데이터에 패딩을 적용해 독립 변수를 전처리해서 문장의 길이가 제각각인 벡터의 크기를 패딩 작업으로 나머지 빈 공간을 0으로 채워준다.

데이터에 패딩을 적용하기 위해 pad_sequences를 import 한다.

In [144]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

pad_sequences() 메소드의 `첫 번째 인수`는 texts_to_sequences() 메소드로 만든 문장을 숫자로 표현한 리스트를 넘겨준다.  
`padding` 속성의 기본값은 'pre'이고 패딩을 시퀀스의 앞쪽에 패딩을 채우고 'post'는 시퀀스의 뒤쪽에 패딩을 채운다.  
`maxlen` 속성은 시퀀스의 최대 길이를 지정한다.

In [150]:
max_length = 500
x_train_sp = pad_sequences(train_sequence, padding='post', maxlen=max_length)
x_test_sp = pad_sequences(test_sequence, padding='post', maxlen=max_length)
print(x_train_sp.shape, x_test_sp.shape)

(1710, 500) (428, 500)


# 모델 만들기

Sequential: 신경망 모델을 구축하기 위해서 레이어를 선형으로 차례차례 쌓는 순차적 모델 구조를 만든다.  
Dense: 모든 입력 뉴런과 출력 뉴런 서로 연결된 완전 연결 레이어로 모델의 최종 출력이나 일반적인 연산에 사용된다.  
Embedding: 텍스트 데이터를 수치형 벡터로 변환하는 임베딩 레이어로 단어 사이의 유사도를 학습하여 고차원의 희소 백터를 저차원의 밀집 벡터로 압축한다.  
Bidirectional: RNN 계열에 적용되며, 데이터를 정방향과 역방향 모드에서 처리하도록 감싸주는 wrapper로 문맥의 앞뒤를 모두 파악해야 하는 자연여 처리에서 성능이 좋다.  
LSTM: RNN의 일종으로, 장기 의존성 문제(긴 시퀀스에서 초기 정보를 잊어버리는 문제)를 해결하기 위해 설계된 레이어로 시계열 데이터나 텍스트 처리에 성능이 좋다.  
Dropout: 학습 시 무작위로 일부 뉴런 끔으로써 과적합을 방지하는 규제 기법이다.  
BatchNormalization: 각 레이어의 출력을 평균을 0, 표준편차가 1이 되도록 정규화하여 정규화하여 학습 속도를 높이고 안정성을 더해준다.

In [151]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, Bidirectional, LSTM, Dropout, BatchNormalization

In [162]:
n_class = y_train.shape[1]
n_class

3

## Bidirectional LSTM

전방향 후방향을 모두 사용하는 양방향 RNN에 LSTM 셀을 가진 모델을 다룬다. RNN이 연쇄된 문장을 다루는 데 유능하지만, 연쇄가 매우 길어질 때는 기울기 소실 문제가 생긴다. 이로 인해 필요한 정보를 멀리까지 보낼 수 있는 방안으로 LSTM이 고안됐다.

단어의 연쇄로 이루어진 데이터가 임베딩 레이어를 거치게 되는데 max_length는 한 문장의 최대 단어 길이로 패딩의 기준이 된다. Bidirectional LSTM의 첫 번째 층은 연쇄로 이루어져 있기 때문에 return_sequences=True로 설정한다.

In [165]:
# 모델 정의 시작: 레이어를 순서대로 하나씩 쌓는 순차적 모델 구조를 생성한다.
model = Sequential([
    # 단어 벡터화 층: 숫자로 변환된 단어 인덱스를 의미를 가지는 밀집 벡터(dence vector)로 바꾼다.
    # input_dim: 단어 사전의 크기
    # output_dim: 단어를 표현할 벡터의 크기(차원)
    # input_shape: 입력되는 문장의 최대 단어 수
    Embedding(input_dim=vocab_size, output_dim=64, input_shape=(max_length,)),
    # 첫 번째 양방향 LSTM 층
    # Bidirectional: 문장을 왼쪽에서 오른쪽으로(정방향), 오른쪽에서 왼쪽으로(역방향) 두 번 읽어 문맥을 양방향에서 파악한다.
    # units: 64개의 유닛을 가진 LSTM 층이다. 긴 문장에서 앞부분의 정보를 끝까지 잘 전달한다.
    # return_sequences=True: 중요한 설정이다. 다음 층도 LSTM이므로, 모든 시점의 출력을 넘겨 연속적인 시퀀스 학습이 가능케 한다.
    Bidirectional(LSTM(units=64, return_sequences=True)),
    # 배치 정규화 층
    # 이전 층의 출력값을 정규하하여 학습 속도를 높이고, 가중치 초기화에 대한 민감도를 줄여 모델을 안정화한다.
    BatchNormalization(),
    # 두 번째 양방향 LSTM 층
    # units: 유닛 개수를 32개로 줄여 정보를 압축한다.
    # return_sequences=True가 없다. 즉, 전체 문장을 다 읽은 후의 최종 요약된 정보만 다음층으로 전달한다.
    Bidirectional(LSTM(units=32)),
    # 학습 시 무작위로 뉴런 20%를 끈다. 특정 뉴런에 과도하게 의존하는 것을 방지하여 과적합을 예방한다.
    Dropout(0.2),
    # 완전 연결 및 출력 층
    # 추출된 특징들을 바탕으로 학습하는 일반적인 신경망 층이다. relu 활성화 함수를 사용해서 비선형성을 추가한다.
    Dense(units=16, activation='relu'),
    # 최종 출력 층: n_class는 분류하려는 카테고리 개수이다.
    # softmax: 출력값을 확류(0 ~ 1 사이, 총합 1)로 변환한다. 가장 높은 확율을 가진 클래스를 최종 예측값으로 선택하게 된다.
    Dense(units=n_class, activation='softmax')
])
# model.summary()